1. INTRODUCTION TO SQL 

# Introduction to SQL for Data Science

SQL (Structured Query Language) is used to manage and manipulate relational databases.

In Data Science, SQL is important for:
- Data extraction
- Data filtering
- Data aggregation
- Understanding relationships between tables
- Performing business analysis

In this notebook, we will demonstrate 10 major SQL concepts using a Retail Store database.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create Tables
cursor.execute("""
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT
);
""")

cursor.execute("""
CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL
);
""")

cursor.execute("""
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    quantity INTEGER
);
""")

# Insert Data
cursor.executemany("INSERT INTO Customers VALUES (?, ?, ?)", [
    (1, 'Rahul', 'Delhi'),
    (2, 'Anita', 'Mumbai'),
    (3, 'Vikram', 'Delhi')
])

cursor.executemany("INSERT INTO Products VALUES (?, ?, ?, ?)", [
    (101, 'Laptop', 'Electronics', 60000),
    (102, 'Mobile', 'Electronics', 20000),
    (103, 'Shoes', 'Fashion', 3000)
])

cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?)", [
    (1, 1, 101, 1),
    (2, 2, 102, 2),
    (3, 1, 103, 3),
    (4, 3, 101, 1)
])

conn.commit()

print("Database Ready")

Database Ready


TOPIC 1: SELECT Statement (with WHERE, ORDER BY, LIMIT)
Definition

SELECT is used to retrieve data from a database.

Problem

Retrieve all customers from Delhi sorted alphabetically.

In [3]:
pd.read_sql_query("""
SELECT name, city
FROM Customers
WHERE city = 'Delhi'
ORDER BY name ASC
LIMIT 5;
""", conn)

,name,city
0,Rahul,Delhi
1,Vikram,Delhi


TOPIC 2: INSERT Statement
Definition

INSERT is used to add new records into a table.

Problem

Add a new customer.

In [5]:
cursor.execute("INSERT INTO Customers VALUES (4, 'Sneha', 'Chennai');")
conn.commit()

pd.read_sql_query("SELECT * FROM Customers;", conn)

,customer_id,name,city
0,1,Rahul,Delhi
1,2,Anita,Mumbai
2,3,Vikram,Delhi
3,4,Sneha,Chennai


TOPIC 3: UPDATE Statement
Definition

UPDATE modifies existing data.

Problem

Update price of Mobile to 22000.

In [7]:
cursor.execute("""
UPDATE Products
SET price = 22000
WHERE product_name = 'Mobile';
""")
conn.commit()

pd.read_sql_query("SELECT * FROM Products;", conn)

,product_id,product_name,category,price
0,101,Laptop,Electronics,60000.0
1,102,Mobile,Electronics,22000.0
2,103,Shoes,Fashion,3000.0


TOPIC 4:DELETE Statement
Definition

DELETE removes records.

Problem

Delete customer with ID 3.

In [9]:
cursor.execute("DELETE FROM Customers WHERE customer_id = 3;")
conn.commit()

pd.read_sql_query("SELECT * FROM Customers;", conn)

,customer_id,name,city
0,1,Rahul,Delhi
1,2,Anita,Mumbai
2,4,Sneha,Chennai


TOPIC 5: JOINS (INNER, LEFT, RIGHT simulated)
Definition

JOIN combines rows from multiple tables.

Problem

Show customer name and product purchased.

INNER JOIN

In [11]:
pd.read_sql_query("""
SELECT C.name, P.product_name
FROM Customers C
INNER JOIN Orders O ON C.customer_id = O.customer_id
INNER JOIN Products P ON O.product_id = P.product_id;
""", conn)

,name,product_name
0,Rahul,Laptop
1,Anita,Mobile
2,Rahul,Shoes


LEFT JOIN

In [13]:
pd.read_sql_query("""
SELECT C.name, P.product_name
FROM Customers C
LEFT JOIN Orders O ON C.customer_id = O.customer_id
LEFT JOIN Products P ON O.product_id = P.product_id;
""", conn)

,name,product_name
0,Rahul,Laptop
1,Rahul,Shoes
2,Anita,Mobile
3,Sneha,None


TOPIC 6: AGGREGATE FUNCTIONS (COUNT, SUM, AVG, MIN, MAX)
Definition

Aggregate functions perform calculations on multiple rows.

Problem

Calculate total revenue.

In [15]:
pd.read_sql_query("""
SELECT SUM(P.price * O.quantity) AS total_revenue
FROM Orders O
JOIN Products P ON O.product_id = P.product_id;
""", conn)

,total_revenue
0,173000.0


TOPIC 7: GROUP BY & HAVING
Definition

GROUP BY groups rows.
HAVING filters grouped data.

Problem

Find total revenue by category where revenue > 50000.

In [17]:
pd.read_sql_query("""
SELECT P.category,
       SUM(P.price * O.quantity) AS revenue
FROM Orders O
JOIN Products P ON O.product_id = P.product_id
GROUP BY P.category
HAVING revenue > 50000;
""", conn)

,category,revenue
0,Electronics,164000.0


TOPIC 8: SUBQUERIES
Definition

A query inside another query.

Problem

Find products with price greater than average price.

In [19]:
pd.read_sql_query("""
SELECT *
FROM Products
WHERE price > (SELECT AVG(price) FROM Products);
""", conn)

,product_id,product_name,category,price
0,101,Laptop,Electronics,60000.0


TOPIC 9: DISTINCT & ALIAS
Definition

DISTINCT removes duplicates.
Alias renames columns.

In [21]:
pd.read_sql_query("""
SELECT DISTINCT city AS Unique_Cities
FROM Customers;
""", conn)

,Unique_Cities
0,Delhi
1,Mumbai
2,Chennai


TOPIC 10: CASE Statement
Definition

CASE adds conditional logic inside SQL.

Problem

Classify products as Expensive or Affordable.

In [23]:
pd.read_sql_query("""
SELECT product_name,
CASE 
    WHEN price > 30000 THEN 'Expensive'
    ELSE 'Affordable'
END AS Price_Category
FROM Products;
""", conn)

,product_name,Price_Category
0,Laptop,Expensive
1,Mobile,Affordable
2,Shoes,Affordable
